In [0]:
from pyspark.sql.functions import *
from datetime import datetime

BATCH_ID = datetime.now().strftime("%Y%m%d%H%M%S")

In [0]:
silver_df = spark.read \
            .format("delta") \
                .load("abfss://silver@saretailsales.dfs.core.windows.net/customers")

In [0]:
display(silver_df)

customer_id,first_name,last_name,dob,email,gender,address,city,state,pincode,country,phone,registration_date,batch_id,source_system,pipeline_name,ingesttime,ingestdate
24,Tanveer,Natt,1973-03-13,noah59@example.net,Male,"88/66 Murthy Street, Lucknow 588191",Bharatpur,Meghalaya,600499,Panama,9723191830,2025-03-11,20260721161046,Retail Sales,03_Bronze_to_Silver,2026-07-21T16:18:20.822Z,2026-07-21
36,Vincent,Chopra,1961-04-27,oyadav@example.com,Female,H.No. 22 Pillai Street Kakinada 621132,Anantapur,Nagaland,258311,Belarus,9902388531,2025-02-23,20260721161046,Retail Sales,03_Bronze_to_Silver,2026-07-21T16:18:20.822Z,2026-07-21
75,Upasna,Luthra,2012-08-02,eghose@example.com,Male,H.No. 85 Thakkar Path Sonipat-773610,Kurnool,Mizoram,924203,India,9390358207,2026-02-05,20260721161046,Retail Sales,03_Bronze_to_Silver,2026-07-21T16:18:20.822Z,2026-07-21
87,Yashica,Karan,1939-02-19,yashvi24@example.com,Female,"61/34, Yadav Marg Jamnagar-269919",Khora,Uttarakhand,59199,Costa Rica,9948058296,2025-10-09,20260721161046,Retail Sales,03_Bronze_to_Silver,2026-07-21T16:18:20.822Z,2026-07-21
102,Falan,Behl,2019-05-24,udeep@example.net,Female,513 Nath Ganj Eluru-784208,Guna,Uttarakhand,523041,Antigua and Barbuda,9380991855,2026-06-01,20260721161046,Retail Sales,03_Bronze_to_Silver,2026-07-21T16:18:20.822Z,2026-07-21
111,Agastya,Suri,1996-07-21,chanabarkha@example.com,Male,35 Sami Road Navi Mumbai 066735,Shivpuri,Kerala,888718,CÃ´te d'Ivoire,9235288824,2024-11-16,20260721161046,Retail Sales,03_Bronze_to_Silver,2026-07-21T16:18:20.822Z,2026-07-21
131,Diya,Bawa,1957-02-27,khatrinetra@example.com,Male,"H.No. 746, Saran, Nanded-404511",Tirupati,Sikkim,184574,Guinea,9687903210,2026-04-24,20260721161046,Retail Sales,03_Bronze_to_Silver,2026-07-21T16:18:20.822Z,2026-07-21
149,Sneha,Chander,1971-04-12,jeet47@example.net,Male,76 Dey Path Gandhidham 595067,Kishanganj,Uttar Pradesh,854034,Guyana,9005598003,2024-08-19,20260721161046,Retail Sales,03_Bronze_to_Silver,2026-07-21T16:18:20.822Z,2026-07-21
153,Yagnesh,Wadhwa,1915-04-14,chaturasandhu@example.com,Female,"H.No. 389, Dayal Chowk Thanjavur-561046",Aizawl,Sikkim,381472,Belize,9584567228,2025-04-11,20260721161046,Retail Sales,03_Bronze_to_Silver,2026-07-21T16:18:20.822Z,2026-07-21
256,Charan,Basak,1944-05-05,bajajvedika@example.net,Male,H.No. 75 Jani Street Haridwar-146449,Agartala,Tamil Nadu,990583,Syria,9528970779,2026-05-07,20260721161046,Retail Sales,03_Bronze_to_Silver,2026-07-21T16:18:20.822Z,2026-07-21


In [0]:
# STATE WISE CUSTOMER COUNT
customer_state_summary = silver_df.groupBy("state") \
                            .count() \
                            .withColumnRenamed("count", "customer_count") \
                            .withColumn("etl_load_timestamp", current_timestamp()) \
                            .withColumn("batch_id", lit(BATCH_ID))

display(customer_state_summary)

state,customer_count,etl_load_timestamp,batch_id
Tripura,36,2026-07-22T14:17:32.293Z,20260722141732
Jharkhand,43,2026-07-22T14:17:32.293Z,20260722141732
Uttar Pradesh,46,2026-07-22T14:17:32.293Z,20260722141732
Tamil Nadu,29,2026-07-22T14:17:32.293Z,20260722141732
Odisha,32,2026-07-22T14:17:32.293Z,20260722141732
Rajasthan,36,2026-07-22T14:17:32.293Z,20260722141732
Telangana,30,2026-07-22T14:17:32.293Z,20260722141732
Assam,44,2026-07-22T14:17:32.293Z,20260722141732
Himachal Pradesh,34,2026-07-22T14:17:32.293Z,20260722141732
Bihar,39,2026-07-22T14:17:32.293Z,20260722141732


In [0]:
customer_state_summary.write \
    .mode("overwrite") \
        .format("delta") \
            .save("abfss://gold@saretailsales.dfs.core.windows.net/customers/customer_state_summary")

In [0]:
# GENDER WISE CUSTOMER COUNT
customer_gender_summary = silver_df.groupBy("gender") \
                            .count() \
                            .withColumnRenamed("count", "customer_count") \
                            .withColumn("etl_load_timestamp", current_timestamp()) \
                            .withColumn("batch_id", lit(BATCH_ID))

display(customer_gender_summary)

gender,customer_count,etl_load_timestamp,batch_id
Male,509,2026-07-22T14:20:13.594Z,20260722141732
Female,488,2026-07-22T14:20:13.594Z,20260722141732


In [0]:
customer_state_summary.write \
    .mode("overwrite") \
        .format("delta") \
            .save("abfss://gold@saretailsales.dfs.core.windows.net/customers/customer_gender_summary")

In [0]:
# CITY WISE CUSTOMER COUNT
customer_city_summary = silver_df.groupBy("city") \
                            .count() \
                            .withColumnRenamed("count", "customer_count") \
                            .withColumn("etl_load_timestamp", current_timestamp()) \
                            .withColumn("batch_id", lit(BATCH_ID))

display(customer_city_summary)

city,customer_count,etl_load_timestamp,batch_id
Shivpuri,5,2026-07-22T14:21:08.497Z,20260722141732
Orai,5,2026-07-22T14:21:08.497Z,20260722141732
Durg,5,2026-07-22T14:21:08.497Z,20260722141732
Kolhapur,7,2026-07-22T14:21:08.497Z,20260722141732
Nanded,2,2026-07-22T14:21:08.497Z,20260722141732
Sikar,5,2026-07-22T14:21:08.497Z,20260722141732
Panvel,4,2026-07-22T14:21:08.497Z,20260722141732
Visakhapatnam,4,2026-07-22T14:21:08.497Z,20260722141732
Thanjavur,6,2026-07-22T14:21:08.497Z,20260722141732
Nashik,1,2026-07-22T14:21:08.497Z,20260722141732


In [0]:
customer_city_summary.write \
    .mode("overwrite") \
        .format("delta") \
            .save("abfss://gold@saretailsales.dfs.core.windows.net/customers/customer_city_summary")

In [0]:
# MONTHLY REGISTRATION CUSTOMER COUNT

silver_df = silver_df.withColumn("year_month", date_format(col("registration_date"), "yyyy-MM"))

customer_monthly_registration = silver_df.groupBy("year_month") \
                            .count() \
                            .withColumnRenamed("count", "customer_count") \
                            .withColumn("etl_load_timestamp", current_timestamp()) \
                            .withColumn("batch_id", lit(BATCH_ID))

display(customer_monthly_registration)

year_month,customer_count,etl_load_timestamp,batch_id
2026-05,41,2026-07-22T14:32:00.274Z,20260722141732
2025-03,45,2026-07-22T14:32:00.274Z,20260722141732
2026-02,34,2026-07-22T14:32:00.274Z,20260722141732
2025-06,42,2026-07-22T14:32:00.274Z,20260722141732
2024-07,17,2026-07-22T14:32:00.274Z,20260722141732
2025-01,29,2026-07-22T14:32:00.274Z,20260722141732
2026-01,44,2026-07-22T14:32:00.274Z,20260722141732
2026-04,39,2026-07-22T14:32:00.274Z,20260722141732
2025-05,41,2026-07-22T14:32:00.274Z,20260722141732
2025-04,35,2026-07-22T14:32:00.274Z,20260722141732


In [0]:
customer_monthly_registration.write \
    .mode("overwrite") \
        .format("delta") \
            .save("abfss://gold@saretailsales.dfs.core.windows.net/customers/customer_monthly_registration")

In [0]:

silver_df = silver_df.withColumn("age", (datediff(current_date(), col("dob")) / 365).cast("int"))

silver_df = silver_df.withColumn("age_group",
    when((col("age") >= 18) & (col("age") <= 29), "18-29")
    .when((col("age") >= 30) & (col("age") <= 39), "30-39")
    .when((col("age") >= 40) & (col("age") <= 49), "40-49")
    .when((col("age") >= 50) & (col("age") <= 59), "50-59")
    .when((col("age") >= 60) & (col("age") <= 69), "60-69")
    .when((col("age") >= 70) & (col("age") <= 79), "70-79")
    .when((col("age") >= 80) & (col("age") <= 89), "80-89")
    .when((col("age") >= 90) & (col("age") <= 99), "90-99")
    .otherwise("100+")
)

customer_age_group_summary = silver_df.groupBy("age_group") \
                                        .count() \
                                        .withColumnRenamed("count", "customer_count") \
                                        .withColumn("etl_load_timestamp", current_timestamp()) \
                                        .withColumn("batch_id", lit(BATCH_ID)) \
                                        .orderBy(col("age_group").asc())

display(customer_age_group_summary)


age_group,customer_count,etl_load_timestamp,batch_id
100+,279,2026-07-22T15:37:06.579Z,20260722153515
18-29,107,2026-07-22T15:37:06.579Z,20260722153515
30-39,71,2026-07-22T15:37:06.579Z,20260722153515
40-49,85,2026-07-22T15:37:06.579Z,20260722153515
50-59,83,2026-07-22T15:37:06.579Z,20260722153515
60-69,95,2026-07-22T15:37:06.579Z,20260722153515
70-79,95,2026-07-22T15:37:06.579Z,20260722153515
80-89,96,2026-07-22T15:37:06.579Z,20260722153515
90-99,86,2026-07-22T15:37:06.579Z,20260722153515


In [0]:
customer_age_group_summary.write \
    .mode("overwrite") \
        .format("delta") \
            .save("abfss://gold@saretailsales.dfs.core.windows.net/customers/customer_age_group_summary")